# 03 — Exploratory data analysis (Stage B)

Label structure and original-image properties of VizWiz-QualityIssues, before any modelling.
All analysis logic lives in `src/eda/`; this notebook only calls it and displays results.
It regenerates `reports/eda.md` and `reports/figures/eda_*.png`.

**Partitions:** train and val only. The test partition is never read during development.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`. Part 2 also needs
`DATA_DIR/image_properties.csv`, produced by
`python scripts/extract_image_properties.py DATA_DIR`; without it, Part 2 is skipped.
No GPU needed.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "eda.md"
FIGURES_DIR = REPO_DIR / "reports" / "figures"

from src.data.splits import MODELLED_FLAWS, get_splits
from src.eda import image_props as ip
from src.eda import labels as lb
from src.eda import report as rp

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(rp.environment_lines()))

In [ ]:
splits = get_splits(DATA_DIR)
train = splits["train"]
print({name: len(splits[name]) for name in ("train", "val")})
sections = []  # (heading, markdown) pairs collected for reports/eda.md

def show(fig, name):
    rp.save(fig, FIGURES_DIR, name)
    display(fig)
    plt.close(fig)

def figure_link(name):
    return f"![{name}](figures/{name}.png)" 

## Part 1 — Labels

### 1.1 Prevalence at >=1 and >=2 votes
Train vs val should match closely; any gap would mean the partitions differ in kind.
UNREC is the unrecognizable vote (>=2 is the dataset authors' "poor image" definition).

In [ ]:
prevalence = lb.prevalence_table(splits)
display(prevalence)
show(rp.plot_prevalence(prevalence), "eda_prevalence")
sections.append(("Prevalence (% of images)",
                 rp.df_to_markdown(prevalence) + "\n\n" + figure_link("eda_prevalence")))

### 1.2 Co-occurrence (train, >=2 votes)
Cell [A, B] = P(B | A) in %. Lift = P(B | A) / P(B): above 1 means the pair occurs together more than chance.
Flaws that share a cause (e.g. low light -> long exposure -> motion blur) can make one flaw look
predictable "by proxy" from another flaw's features.

In [ ]:
conditional, joint = lb.cooccurrence(train)
lift = lb.lift_matrix(train)
display(conditional, lift)
show(rp.plot_matrix(conditional, "P(B | A), %"), "eda_cooccurrence")
show(rp.plot_matrix(lift, "Lift P(B|A)/P(B)", fmt="{:.2f}", cmap="RdBu_r"), "eda_lift")
sections.append(("Flaw co-occurrence (train, >=2 votes)",
                 "P(B | A), %:\n\n" + rp.df_to_markdown(conditional)
                 + "\n\nLift P(B | A) / P(B):\n\n" + rp.df_to_markdown(lift)
                 + "\n\nJoint counts:\n\n" + rp.df_to_markdown(joint)
                 + "\n\n" + figure_link("eda_cooccurrence") + "\n\n" + figure_link("eda_lift")))

### 1.3 Vote distributions and annotator agreement (train)
Fleiss' kappa per label from the 5-rater vote counts (1 = perfect agreement, 0 = chance level).
"borderline 2-3 %" = images whose >=2 label would flip with a single rater changing their mind.
A preliminary view; the full human ceiling is Stage G.

In [ ]:
votes_dist = lb.vote_distribution(train)
agreement = lb.agreement_table(train)
display(votes_dist, agreement)
show(rp.plot_vote_distribution(votes_dist), "eda_votes")
sections.append(("Vote distributions (train, % of images)",
                 rp.df_to_markdown(votes_dist) + "\n\n" + figure_link("eda_votes")))
sections.append(("Annotator agreement (train)", rp.df_to_markdown(agreement)))

### 1.4 Flaws and recognisability (train)
Which flaws actually make an image unusable, and does unusability grow with the number of flaws?

In [ ]:
by_count = lb.unrecognisable_by_flaw_count(train)
by_flaw = lb.unrecognisable_by_flaw(train)
display(by_count, by_flaw)
sections.append(("Unrecognisable rate by number of flaws (train, >=2 votes)",
                 rp.df_to_markdown(by_count)))
sections.append(("Unrecognisable rate by flaw (train, >=2 votes)", rp.df_to_markdown(by_flaw)))

## Part 2 — Original-image properties

Read from the original zips by `scripts/extract_image_properties.py` (headers only, raw pixel
dimensions, no EXIF rotation applied).

In [ ]:
props_path = DATA_DIR / "image_properties.csv"
props = None
if props_path.exists():
    props = ip.load_properties(props_path)
    props = props[props["partition"].isin(["train", "val"])]
    print(props["partition"].value_counts().to_dict())
else:
    print(f"{props_path} not found - Part 2 skipped. Run scripts/extract_image_properties.py first.")

### 2.1 Native resolution -> classical@512 decision
Pre-registered rule (docs/decisions.md): keep 512 unless more than 10% of originals have a long edge
below 512 (they would be upsampled, which smooths them and fakes blur).

In [ ]:
if props is not None:
    percentiles, below = ip.resolution_summary(props)
    verdicts = [ip.resolution_verdict(props, r)[1] for r in ip.CLASSICAL_RESOLUTIONS]
    display(percentiles.to_frame("long edge (px)"), below)
    for v in verdicts:
        print(v)
    show(rp.plot_long_edge(props), "eda_long_edge")
    sections.append(("Native resolution (train + val)",
                     rp.df_to_markdown(percentiles.to_frame("long edge (px)"))
                     + "\n\n" + "\n".join(f"- {k}: {v}" for k, v in below.items())
                     + "\n\nPre-registered upsampling rule:\n\n"
                     + "\n".join(f"- {v}" for v in verdicts)
                     + "\n\n" + figure_link("eda_long_edge")))

### 2.2 Aspect ratio and file size

In [ ]:
if props is not None:
    sizes = rp.file_size_summary(props)
    shapes = ip.shape_by_orientation(props)
    display(sizes, shapes)
    show(rp.plot_aspect(props), "eda_aspect")
    sections.append(("Aspect ratio and file size (train + val)",
                     "Raw pixel shape by EXIF orientation:\n\n" + rp.df_to_markdown(shapes)
                     + "\n\n" + rp.df_to_markdown(sizes)
                     + "\n\n" + figure_link("eda_aspect")))

### 2.3 ROT shortcut check (train)
Can ROT be predicted from raw pixel shape alone, with no image content? AP is compared with ROT
prevalence (the AP of a random guess). The "orientation 1 only" row removes the EXIF-tag-6 images,
so it shows whether shape carries signal beyond standing in for the tag.

In [ ]:
if props is not None:
    train_props = props[props["partition"] == "train"]
    train_votes = lb.vote_matrix(train)
    rot_table = ip.rot_shortcut_table(train_props, train_votes)
    rot_scores = ip.rot_shortcut_scores(train_props, train_votes)
    display(rot_table, rot_scores)
    sections.append(("ROT shortcut check (train, ROT >=2 votes)",
                     "ROT rate by EXIF orientation x raw pixel shape:\n\n"
                     + rp.df_to_markdown(rot_table)
                     + "\n\nGeometry-only cues scored as ROT predictors:\n\n"
                     + rp.df_to_markdown(rot_scores)))

## Write the report

In [ ]:
out = rp.write_report(REPORT_PATH, sections)
print("Report written to", out)